In [1]:
"""!pip install --upgrade pylace
"""

'!pip install --upgrade pylace\n'

In [2]:
# lace.ipynb – Cell 1: Setup

import pandas as pd
pd.set_option("display.notebook_repr_html", False)

import numpy as np

import lace
from lace.plot import diagnostics  # for convergence plots

print("lace version:", lace.__version__)


lace version: 0.9.0


In [3]:
# lace.ipynb – Cell 2: Load cleaned data for Lace

import os

print("Files in CWD:", os.listdir())

df = pd.read_csv("cleaned_data.csv")  # you saved index=False, so no index_col here
print("Raw shape:", df.shape)
df.head()


Files in CWD: ['.git', '.gitignore', 'cleaned_data.csv', 'data cleaning.ipynb', 'lace.ipynb', 'Lateral Data 2014-2022.xlsx', 'LICENSE', 'README.md', 'try copy.ipynb', 'try.ipynb', 'uncleaned_data.csv']
Raw shape: (546, 115)


  Date of Surgery  Sex  Age       Case/Type of Surgery  Perc screws?  Open  \
0       1/28/2014    0   63  llif l4-l5 w/ perc screws             1     0   
1       1/28/2014    0   55                 llif l2-l5             0     0   
2       1/29/2014    1   74                 llif l2-l3             0     0   
3       2/10/2014    1   55                llif l3-l5              0     0   
4       2/24/2014    1   67                 llif l4-l5             0     0   

   Open Check V2  Standalone XLIF Check  \
0              0                      0   
1              0                      1   
2              0                      1   
3              0                      1   
4              0                      1   

    Retroperitoneal Approach (LLIF ± ALIF)  Anterior + Posterior Apporoach  \
0                                        0                               1   
1                                        1                               0   
2                                     

In [4]:
# lace.ipynb – Cell 3: Light cleanup for Lace

df_lace = df.copy()

drop_cols = []

# 1) Validation column if present
if "VALIDATION COLUMN" in df_lace.columns:
    drop_cols.append("VALIDATION COLUMN")

# 2) Duplicate complication columns with ".1" suffix
drop_cols.extend([c for c in df_lace.columns if c.endswith(".1")])

# 3) Date-like columns (keep the derived durations instead)
date_like = [
    "Date of Surgery",
    "post-op date",
    "most recent date",
    "discharge date",
    "Last follow-up date",
]
drop_cols.extend([c for c in date_like if c in df_lace.columns])

# Make unique
drop_cols = list(dict.fromkeys(drop_cols))

print("Dropping columns:", drop_cols)
df_lace = df_lace.drop(columns=drop_cols)
print("Shape after Lace cleanup:", df_lace.shape)

df_lace.head()


Dropping columns: ['VALIDATION COLUMN', 'acute thigh paresthesia (immediate post op).1', 'transient paresthesia (post op clinic note).1', 'psoas hematoma.1', 'abdominal hernia (post op clinic note).1', 'Date of Surgery', 'post-op date', 'most recent date', 'discharge date', 'Last follow-up date']
Shape after Lace cleanup: (546, 105)


   Sex  Age       Case/Type of Surgery  Perc screws?  Open  Open Check V2  \
0    0   63  llif l4-l5 w/ perc screws             1     0              0   
1    0   55                 llif l2-l5             0     0              0   
2    1   74                 llif l2-l3             0     0              0   
3    1   55                llif l3-l5              0     0              0   
4    1   67                 llif l4-l5             0     0              0   

   Standalone XLIF Check   Retroperitoneal Approach (LLIF ± ALIF)  \
0                      0                                        0   
1                      1                                        1   
2                      1                                        1   
3                      1                                        1   
4                      1                                        1   

   Anterior + Posterior Apporoach  Osteotomies (yes/no)  ... alif_text  \
0                               1               

In [5]:
# lace.ipynb – Cell: Drop constant columns

import numpy as np

# Count unique values (including NaN treated as a category)
nunique = df_lace.nunique(dropna=False)

constant_cols = nunique[nunique <= 1].index.tolist()
print("Constant columns to drop:", constant_cols)

df_lace_model = df_lace.drop(columns=constant_cols)
print("Original shape:", df_lace.shape)
print("Modeling shape:", df_lace_model.shape)


Constant columns to drop: ['T12-L1', 'L1-L2', 'L2-L3', 'L3-L4', 'L4-L5', 'L5-S1', 'levels_fused_count', 'construct_span_levels', 'thoracolumbar_junction', 'upper_lumbar', 'lower_lumbar', 'lumbosacral']
Original shape: (546, 105)
Modeling shape: (546, 93)


In [6]:
from lace import Codebook

codebook = Codebook.from_df("spine_lateral_llif", df_lace_model)
codebook


PanicException: called `Result::unwrap()` on an `Err` value: SigmaTooLow { sigma: 0.0 }

In [ ]:
from lace import Engine

engine = Engine.from_df(df_lace_model, codebook=codebook)
engine.update(1_000)   # start small to test


TypeError: argument 'codebook': 'DataFrame' object cannot be converted to 'Codebook'